In [2]:
import numpy as np
import pandas as pd

In [3]:
data=pd.read_csv('e_waste_dataset_with_profit.csv')

In [4]:
data.head()

,Item,Category,Gold,Silver,Platinum,Rhodium,Nickel,Tin,Lithium,Aluminum,Carbon,Profit ($)
0,iPhone 11,Cat3,3.58,2.95,1.73,8.92,1.91,1.01,1.82,1.27,9.51,270.34
1,Toaster,Cat2,7.21,4.31,6.21,5.63,9.59,7.65,0.51,3.03,4.22,689.75
2,Speaker,Cat4,8.91,5.09,2.42,7.70,1.09,1.49,7.42,3.63,8.83,570.43
3,Microwave Oven,Cat2,2.62,3.84,2.98,7.66,9.41,2.25,7.84,6.18,6.36,290.78
4,Air Conditioner,Cat1,3.47,3.89,6.20,4.35,5.07,8.65,8.62,0.82,5.53,505.16


In [5]:
print(data.info())
print(data.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Item        10000 non-null  object 
 1   Category    10000 non-null  object 
 2   Gold        10000 non-null  float64
 3   Silver      10000 non-null  float64
 4   Platinum    10000 non-null  float64
 5   Rhodium     10000 non-null  float64
 6   Nickel      10000 non-null  float64
 7   Tin         10000 non-null  float64
 8   Lithium     10000 non-null  float64
 9   Aluminum    10000 non-null  float64
 10  Carbon      10000 non-null  float64
 11  Profit ($)  10000 non-null  float64
dtypes: float64(10), object(2)
memory usage: 937.6+ KB
None
               Gold        Silver      Platinum       Rhodium        Nickel  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean       5.058206      5.001442      5.055381      5.086352      5.032354   
std        2.863858  

In [6]:
print(data.isnull().sum())

Item          0
Category      0
Gold          0
Silver        0
Platinum      0
Rhodium       0
Nickel        0
Tin           0
Lithium       0
Aluminum      0
Carbon        0
Profit ($)    0
dtype: int64


In [7]:
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())

In [8]:
data['Profit ($)'] = data['Profit ($)'].astype(float)

In [9]:
print(data.columns)

Index(['Item', 'Category', 'Gold', 'Silver', 'Platinum', 'Rhodium', 'Nickel',
       'Tin', 'Lithium', 'Aluminum', 'Carbon', 'Profit ($)'],
      dtype='object')


In [10]:
data.rename(columns={'old_name':'new_name'}, inplace=True)

In [11]:
numeric_data = data.select_dtypes(include=[np.number])
Q1 = numeric_data.quantile(0.25)
Q3 = numeric_data.quantile(0.75)
IQR = Q3 - Q1

data = data[~((numeric_data < (Q1 - 1.5 * IQR)) | (numeric_data > (Q3 + 1.5 * IQR))).any(axis=1)]

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numeric_cols = data.select_dtypes(include=[np.number]).columns
data_scaled = scaler.fit_transform(data[numeric_cols])

In [13]:
X = data.drop('Profit ($)', axis=1)
y = data['Profit ($)']

In [14]:
from sklearn.preprocessing import OneHotEncoder
onehotencoder = OneHotEncoder()
categorical_cols = X.select_dtypes(include=['object']).columns


In [15]:
from sklearn.compose import ColumnTransformer
ct=ColumnTransformer(transformers=[('encoder', OneHotEncoder(), categorical_cols)], remainder='passthrough')
X = ct.fit_transform(X)

In [16]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X = imputer.fit_transform(X)

In [17]:
y

0       270.34
1       689.75
2       570.43
3       290.78
4       505.16
         ...  
9995    132.55
9996    611.51
9997    583.98
9998    547.01
9999    703.01
Name: Profit ($), Length: 10000, dtype: float64

In [18]:
X

array([[0.  , 0.  , 0.  , ..., 1.82, 1.27, 9.51],
       [0.  , 0.  , 0.  , ..., 0.51, 3.03, 4.22],
       [0.  , 0.  , 0.  , ..., 7.42, 3.63, 8.83],
       ...,
       [1.  , 0.  , 0.  , ..., 4.38, 7.93, 4.67],
       [0.  , 0.  , 0.  , ..., 3.16, 9.93, 7.28],
       [0.  , 0.  , 1.  , ..., 7.85, 7.22, 6.68]])

In [19]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

if 'Category' in data.columns:
    data['Category'] = labelencoder.fit_transform(data['Category'].astype(str))

In [20]:
print("Data shape:", data.shape)
print("Data types:\n", data.dtypes)

Data shape: (10000, 12)
Data types:
 Item           object
Category        int32
Gold          float64
Silver        float64
Platinum      float64
Rhodium       float64
Nickel        float64
Tin           float64
Lithium       float64
Aluminum      float64
Carbon        float64
Profit ($)    float64
dtype: object


In [78]:
data['Recovery_Status'] = (data['Profit ($)'] > 300).astype(int)

In [79]:
print(data[['Profit ($)', 'Recovery_Status']].head())

   Profit ($)  Recovery_Status
0      270.34                0
1      689.75                1
2      570.43                1
3      290.78                0
4      505.16                1


In [82]:
data.head()

,Item,Category,Gold,Silver,Platinum,Rhodium,Nickel,Tin,Lithium,Aluminum,Carbon,Profit ($),Total_Metals,Avg_Metal_Value,Profit_per_Metal,Precious_Ratio,Recovery_Status
0,iPhone 11,2,3.58,2.95,1.73,8.92,1.91,1.01,1.82,1.27,9.51,270.34,26.17,3.738571,10.330149,0.406955,0
1,Toaster,1,7.21,4.31,6.21,5.63,9.59,7.65,0.51,3.03,4.22,689.75,36.84,5.262857,18.722856,0.321390,1
2,Speaker,3,8.91,5.09,2.42,7.70,1.09,1.49,7.42,3.63,8.83,570.43,32.58,4.654286,17.508594,0.310620,1
3,Microwave Oven,1,2.62,3.84,2.98,7.66,9.41,2.25,7.84,6.18,6.36,290.78,42.68,6.097143,6.813027,0.249297,0
4,Air Conditioner,0,3.47,3.89,6.20,4.35,5.07,8.65,8.62,0.82,5.53,505.16,39.24,5.605714,12.873598,0.268858,1


In [83]:
data

,Item,Category,Gold,Silver,Platinum,Rhodium,Nickel,Tin,Lithium,Aluminum,Carbon,Profit ($),Total_Metals,Avg_Metal_Value,Profit_per_Metal,Precious_Ratio,Recovery_Status
0,iPhone 11,2,3.58,2.95,1.73,8.92,1.91,1.01,1.82,1.27,9.51,270.34,26.17,3.738571,10.330149,0.406955,0
1,Toaster,1,7.21,4.31,6.21,5.63,9.59,7.65,0.51,3.03,4.22,689.75,36.84,5.262857,18.722856,0.321390,1
2,Speaker,3,8.91,5.09,2.42,7.70,1.09,1.49,7.42,3.63,8.83,570.43,32.58,4.654286,17.508594,0.310620,1
3,Microwave Oven,1,2.62,3.84,2.98,7.66,9.41,2.25,7.84,6.18,6.36,290.78,42.68,6.097143,6.813027,0.249297,0
4,Air Conditioner,0,3.47,3.89,6.20,4.35,5.07,8.65,8.62,0.82,5.53,505.16,39.24,5.605714,12.873598,0.268858,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Fridge,0,1.99,2.37,0.62,2.23,4.72,6.75,3.48,7.98,1.36,132.55,27.14,3.877143,4.883935,0.105011,0
9996,Iron,1,8.57,8.68,3.46,5.25,2.78,0.78,9.72,4.46,5.69,611.51,32.14,4.591429,19.026447,0.271002,1
9997,Air Conditioner,0,9.30,6.22,2.30,7.07,7.94,5.56,4.38,7.93,4.67,583.98,39.85,5.692857,14.654454,0.235132,1
9998,Stove,0,6.13,6.74,4.51,9.03,8.76,9.45,3.16,9.93,7.28,547.01,52.12,7.445714,10.495203,0.259785,1


In [84]:
print(data.head())

              Item  Category  Gold  Silver  Platinum  Rhodium  Nickel   Tin  \
0        iPhone 11         2  3.58    2.95      1.73     8.92    1.91  1.01   
1          Toaster         1  7.21    4.31      6.21     5.63    9.59  7.65   
2          Speaker         3  8.91    5.09      2.42     7.70    1.09  1.49   
3   Microwave Oven         1  2.62    3.84      2.98     7.66    9.41  2.25   
4  Air Conditioner         0  3.47    3.89      6.20     4.35    5.07  8.65   

   Lithium  Aluminum  Carbon  Profit ($)  Total_Metals  Avg_Metal_Value  \
0     1.82      1.27    9.51      270.34         26.17         3.738571   
1     0.51      3.03    4.22      689.75         36.84         5.262857   
2     7.42      3.63    8.83      570.43         32.58         4.654286   
3     7.84      6.18    6.36      290.78         42.68         6.097143   
4     8.62      0.82    5.53      505.16         39.24         5.605714   

   Profit_per_Metal  Precious_Ratio  Recovery_Status  
0         10.330149

In [35]:
data['Total_Metals'] = data['Platinum'] + data['Rhodium'] + data['Nickel'] + data['Tin'] + data['Lithium'] + data['Aluminum'] + data['Carbon']

In [36]:
data['Avg_Metal_Value'] = data[['Platinum','Rhodium','Nickel','Tin','Lithium','Aluminum','Carbon']].mean(axis=1)

In [37]:
data['Profit_per_Metal'] = data['Profit ($)'] / data['Total_Metals']

In [39]:
data['Recovery_Status'] = (data['Recovery_Status'] == 'Recoverable').astype(int)

In [40]:
data['Precious_Ratio'] = (data['Platinum'] + data['Rhodium']) / data['Total_Metals']

In [85]:
data.head()

,Item,Category,Gold,Silver,Platinum,Rhodium,Nickel,Tin,Lithium,Aluminum,Carbon,Profit ($),Total_Metals,Avg_Metal_Value,Profit_per_Metal,Precious_Ratio,Recovery_Status
0,iPhone 11,2,3.58,2.95,1.73,8.92,1.91,1.01,1.82,1.27,9.51,270.34,26.17,3.738571,10.330149,0.406955,0
1,Toaster,1,7.21,4.31,6.21,5.63,9.59,7.65,0.51,3.03,4.22,689.75,36.84,5.262857,18.722856,0.321390,1
2,Speaker,3,8.91,5.09,2.42,7.70,1.09,1.49,7.42,3.63,8.83,570.43,32.58,4.654286,17.508594,0.310620,1
3,Microwave Oven,1,2.62,3.84,2.98,7.66,9.41,2.25,7.84,6.18,6.36,290.78,42.68,6.097143,6.813027,0.249297,0
4,Air Conditioner,0,3.47,3.89,6.20,4.35,5.07,8.65,8.62,0.82,5.53,505.16,39.24,5.605714,12.873598,0.268858,1


In [86]:
data.to_csv('cleaned_e_waste_dataset.csv', index=False)